# Diagnose change in ocean-driven SST over time

In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar

# import gsw

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## funcs

In [ ]:
def get_nino_merged(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(190, 270))
    return x.sel(idx).mean(["longitude", "latitude"])


def get_Te(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(230, 280))
    return x.sel(idx).mean(["longitude", "latitude"])


def get_Tw(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(180, 230))
    return x.sel(idx).mean(["longitude", "latitude"])

## Load data

In [ ]:
Th_3d = src.utils.load_cesm_indices_3d()

In [ ]:
## load indices
Th = src.utils.load_cesm_indices()
Th["h_mg"] = Th_3d["h"]
Th["h_w_mg"] = Th_3d["h_w"]

## load 3d indices
# Th_3d = src.utils.load_cesm_indices()

In [ ]:
forced, anom = src.utils.load_consolidated()

## get NHF in Niño regions
nhf_3 = src.utils.reconstruct_wrapper(
    anom[["nhf", "nhf_comp"]], fn=src.utils.get_nino3
)["nhf"]
nhf_34 = src.utils.reconstruct_wrapper(
    anom[["nhf", "nhf_comp"]], fn=src.utils.get_nino34
)["nhf"]
nhf_m = src.utils.reconstruct_wrapper(anom[["nhf", "nhf_comp"]], fn=get_nino_merged)[
    "nhf"
]
nhf_e = src.utils.reconstruct_wrapper(anom[["nhf", "nhf_comp"]], fn=get_Te)["nhf"]
T_m = src.utils.reconstruct_wrapper(anom[["sst", "sst_comp"]], fn=get_nino_merged)[
    "sst"
]

## convert to units of K/mo
sec_per_mo = 8.64e4 * 30
sec_per_yr = 12 * sec_per_mo
rho = 1.02e3
Cp = 4.2e3
H = 50

Q_34 = nhf_34 * sec_per_yr / (rho * Cp * H)
Q_3 = nhf_3 * sec_per_yr / (rho * Cp * H)
Q_m = nhf_m * sec_per_yr / (rho * Cp * H)
Q_e = nhf_e * sec_per_yr / (rho * Cp * H)

## merge with Th data
Th["Q_34"] = Q_34.sel(time=Th.time)
Th["Q_3"] = Q_3.sel(time=Th.time)
Th["Q_e"] = Q_e.sel(time=Th.time)
Th["Q_m"] = Q_m.sel(time=Th.time)
Th["T_m"] = T_m.sel(time=Th.time)

#### window data

In [ ]:
## window
Th = src.utils.get_windowed(Th, stride=120, window_size=480)

## remove median
Th = Th.groupby("time.month") - Th.groupby("time.month").median(["time", "member"])

#### Get h with T-dependence removed

In [ ]:
## specify var to remove
VARR = "T_3"

## remove SST dependence
for h_var in tqdm.tqdm(["h", "h_w", "h_mg", "h_w_mg"]):
    Th[f"{h_var}_hat"] = src.utils.remove_sst_dependence_v2(
        Th=Th,
        h_var=h_var,
        T_var=f"{VARR}",
        dims=["time", "member"],
    )

## compute

In [ ]:
def get_spaghetti(idx, data, peak_month, event_type=None, q=0.75, is_warm=True):
    """
    Get hovmoller composite based on specified:
    - data: used to compute index/make composite
    - peak_month: month to center composite on
    - q: quantile threshold for composite
    """

    ## handle warm/cold case
    if is_warm:
        kwargs = dict(q=q, check_cutoff=lambda x, cut: x > cut)
    else:
        kwargs = dict(q=1 - q, check_cutoff=lambda x, cut: x < cut)

    ## kwargs for composite
    kwargs = dict(
        kwargs,
        avg=False,
        peak_month=peak_month,
        idx=idx,
        data=data,
        event_type=event_type,
    )

    ## composite of projected data
    spag = src.utils.make_composite(**kwargs)

    return spag

In [ ]:
## empty arrays to hold results
spags_warm = []
spags_cold = []


## loop thru years
for year in tqdm.tqdm(Th.year):

    ## get data
    data_y = Th.sel(year=year)

    ## shared args
    kwargs = dict(idx=data_y["T_34"], data=data_y, peak_month=1, q=0.75)

    ## compute composites
    spags_warm.append(get_spaghetti(is_warm=True, **kwargs))
    spags_cold.append(get_spaghetti(is_warm=False, **kwargs))

## put in xarray
spags_warm = xr.concat(spags_warm, dim=Th.year)
spags_cold = xr.concat(spags_cold, dim=Th.year)

Compute GR

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(6, 6), layout="constrained")

for j, lag in enumerate([-6, 0]):

    for ax, spags_, s in zip(axs[j], [spags_warm, spags_cold], [1, -1]):
        ax.scatter(
            s * spags_["T_3"].sel(lag=lag).isel(year=0),
            s * spags_["h_hat"].sel(lag=lag).isel(year=0),
            s=7,
            # 2*spags_warm["h"].sel(lag=-6).isel(year=0)-spags_warm["h_w"].sel(lag=-6).isel(year=0),
        )
        ax.axvline(0, c="k")
        ax.axhline(0, c="k")

for ax in axs[0]:
    ax.set_xticks([])
for ax in axs[:, 1]:
    ax.set_yticks([])
src.utils.set_lims(axs.flatten())
axs[0, 0].set_title("warm")
axs[0, 1].set_title("cold")
plt.show()

In [ ]:
def get_dSST(spags, norm=False, t0=-6, t1=0, thresh=0.25):
    """get normalized"""

    ## normalize
    if norm:

        ## get sign
        sign = np.sign(spags.sel(lag=0).mean("sample"))

        ## subset for same sign
        same_sign = (spags * sign) > thresh
        spags_ = spags.where(same_sign, other=np.nan)

        ## differentiate
        ddt_SST = spags_.differentiate("lag")

        ## normalize and integrate
        dSST = sign * (ddt_SST / spags_).sel(lag=slice(t0, t1)).sum("lag")

    else:
        ## get dSST
        dSST = spags.sel(lag=t1) - spags.sel(lag=t0)

    return dSST

In [ ]:
## get normal
kwargs = dict(norm=False, t0=-6, t1=0, thresh=0)
dSST_warm = get_dSST(spags_warm, **kwargs)
dSST_cold = get_dSST(spags_cold, **kwargs)


## norm kwargs
kwargs = dict(norm=True, t0=-6, t1=-1, thresh=0.75)
dSST_warm_norm = get_dSST(spags_warm, **kwargs)
dSST_cold_norm = get_dSST(spags_cold, **kwargs)

for [dSST_w, dSST_c] in [
    [dSST_warm, dSST_cold],
    [dSST_warm_norm, dSST_cold_norm],
]:

    fig, axs = plt.subplots(1, 3, figsize=(8, 2.5), layout="constrained")

    ## plot data
    for ax, PLOT_VAR in zip(axs, ["T_34", "T_m", "T_3"]):
        ax.plot(dSST_w.year, dSST_w[PLOT_VAR].mean("sample"), c="r")
        ax.plot(dSST_w.year, -dSST_c[PLOT_VAR].mean("sample"), c="b")

    ## formatting
    # ax.axhline(0, ls="--", c='gray', lw=1)
    src.utils.add_vticks(axs, xticks=[2010, 2050], xlines=[2010, 2050])
    # src.utils.set_lims(axs)
    # for ax in axs[1:]:
    #     ax.set_yticks([])
    plt.show()

In [ ]:
## specify plot var
PLOT_VAR = "T_3"

for s in [spags_warm, -spags_cold]:

    fig, ax = plt.subplots(figsize=(3, 2.5))
    ax.plot(s.lag, s.isel(year=0)[PLOT_VAR].mean("sample"))
    ax.plot(s.lag, s.sel(year=2010)[PLOT_VAR].mean("sample"))
    ax.plot(s.lag, s.sel(year=2050)[PLOT_VAR].mean("sample"))
    src.utils.add_vticks([ax], xticks=[-6], xlines=[-6, 0])
    plt.show()
# plt.plot(spags_warm.lag, -spags_cold.isel(year=0)["T_3"].mean("sample"))

## Fit LIM to warm/cold

### Fitting func

In [ ]:
def regress_pinv2(X, x_vars, y_var):
    """do nonlinear regression"""

    ## prep data
    prep = lambda x: x.transpose("year", "lag", "sample", ...)

    X_ = prep(X[x_vars].to_dataarray(dim="v")).transpose(..., "v", "sample")
    Y_ = prep(X[y_var])

    ## empty array to hold results
    m = xr.DataArray(
        coords={"year": X.year, "lag": X.lag, "v": x_vars},
        dims=["year", "lag", "v"],
    )

    ## do regression
    X_pinv = np.linalg.pinv(X_.values)
    m.values = np.einsum("yli,ylij->ylj", Y_.values, X_pinv)

    return m

### Prep data

In [ ]:
def prep_data(X):
    """prep data to fit RO"""

    ## differentiate wrt time
    ddt_X = 12 * X.differentiate("lag").rename({v: f"ddt_{v}" for v in list(X)})

    ## merge data
    Z = xr.merge([X, ddt_X, xr.ones_like(X["T_3"]).rename("ones")])

    ## get dataset
    return Z.sel(lag=slice(-8, 3)).isel(year=slice(None, -1))

In [ ]:
## prep data
Xw = prep_data(spags_warm)
Xc = prep_data(spags_cold)

### Test: comparison to growth rate calc.

In [ ]:
## specify vars
TVAR = "T_34"

## feature variables
# XVARS = [TVAR, "ones"]
XVARS = [TVAR]

## shared args
kwargs = dict(x_vars=XVARS, y_var=f"ddt_{TVAR}")

## fit coefficients
Lw = regress_pinv2(Xw, **kwargs)
Lc = regress_pinv2(Xc, **kwargs)

In [ ]:
sel = lambda x: delta(x.sel(lag=slice(-6, -1)).mean("lag"))
Lw_ = sel(Lw)
Lc_ = sel(Lc)

fig, axs = plt.subplots(1, 2, figsize=(5.5, 2.5), layout="constrained")

for ax, v in zip(axs, Lw.v.values):
    ax.plot(Lw_.year, Lw_.sel(v=v), c="r")
    ax.plot(Lc_.year, Lc_.sel(v=v), c="b")
    ax.axhline(0, ls="--", c="gray", lw=1)
    ax.set_title(v)

# src.utils.set_lims(axs)
# axs[1].set_yticks([])
plt.show()

### Full LIM

In [ ]:
## specify vars
TVAR = "T_3"
HVAR = "h_w_mg"

## shared args
kwargs = dict(x_vars=[TVAR, HVAR, "ones"], y_var=f"ddt_{TVAR}")
kwargs_h = dict(x_vars=[TVAR, HVAR, "ones"], y_var=f"ddt_{HVAR}")

## fit coefficients
Lw = regress_pinv2(Xw, **kwargs)
Lc = regress_pinv2(Xc, **kwargs)

Lw_h = -regress_pinv2(Xw, **kwargs_h)
Lc_h = -regress_pinv2(Xc, **kwargs_h)

In [ ]:
delta = lambda x: x - x.isel(year=0)

sel = lambda x: delta(x.sel(lag=slice(-7, 0)).mean("lag"))
Lw_ = sel(Lw)
Lc_ = sel(Lc)

for [Lw_plot, Lc_plot], labels in zip(
    [[Lw, Lc], [Lw_h, Lc_h]],
    [[r"$R$", r"$F_1$", "const."], [r"$F_2$", r"$\varepsilon$", "const."]],
):

    ## get data to plot
    Lw_ = sel(Lw_plot)
    Lc_ = sel(Lc_plot)

    for v, label in zip([TVAR, HVAR, "ones"], labels):

        fig, ax = plt.subplots(figsize=(3, 2.5))
        ax.plot(Lw_.year, Lw_.sel(v=v), c="r")
        ax.plot(Lc_.year, Lc_.sel(v=v), c="b")
        ax.axhline(0, ls="--", c="gray", lw=1)
        ax.set_title(label)
        plt.show()